# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamal1osama/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane as an ML task (type)

**Chosen Lane:** **Lane 2: Refresh / Content Opportunity Scoring**

### Task Type: Learning-to-Rank / Prioritized Scoring (via Supervised Classification with Calibrated Probabilities)

- **Why this task type:**
  The core operational question is: *"Which 20 to 50 URLs from our content catalog should our editorial team review and refresh first during the next sprint?"*
  - It is **not unconstrained classification**: In our catalog of 30,000 items, over 16,000 pages show downward trend indicators. Simply outputting a binary 0/1 prediction leaves editors with thousands of "positive" pages and no way to allocate their finite 10–20 hours of review bandwidth.
  - It is **not unsupervised clustering**: Clustering can identify structural archetypes, but it does not prioritize urgency or predict which high-demand pages are experiencing traffic erosion.
  - It **is a ranking and opportunity scoring problem**: We train a supervised model to estimate a calibrated probability of decay risk $P(\text{decline} \mid X)$ from historical observable signals, scale it by exposure stakes (impressions and Page 1 position risk), and generate a continuous priority score $S \in [0, 100]$. The final output is an ordered action queue partitioned into sprint-sized review cohorts.

### Action the Output Supports
The primary consumer is an **SEO content strategist / senior editor**. The editor opens the ranked queue filtered by client, inspects the top 20 pages alongside their diagnostic reason codes (e.g., `declining_with_demand`, `page_one_decay_risk`, `stale_visible_page`, `low_ctr_visible_page`), verifies current search engine results pages (SERPs), and takes concrete on-page actions: updating outdated statistics, expanding thin sections, re-aligning headings to shifted search intent, or improving title/meta descriptions.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Robust path resolution for local, repo root, and Google Colab environments
data_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

# Filter to eligible candidate pool: mature content with active search visibility
eligible_mask = (df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)
df_eligible = df[eligible_mask].copy()

print("=== Task Formulation: Content Refresh Ranking Queue ===")
print(f"Total Content Items in Catalog: {len(df):,}")
print(f"Eligible Candidates (age >= 90d, impressions > 0): {len(df_eligible):,}")
print(f"Down-trending items in eligible pool: {(df_eligible['trend_direction'] == 'down').sum():,} ({(df_eligible['trend_direction'] == 'down').mean():.1%})")
print("\nOperational Constraint: Editorial teams can realistically review only 10-20 items/week.")
print("Goal of ML: Score and rank candidate items into an ordered top-K queue with diagnostic reason codes.")



=== Task Formulation: Content Refresh Ranking Queue ===
Total Content Items in Catalog: 30,000
Eligible Candidates (age >= 90d, impressions > 0): 30,000
Down-trending items in eligible pool: 16,262 (54.2%)

Operational Constraint: Editorial teams can realistically review only 10-20 items/week.
Goal of ML: Score and rank candidate items into an ordered top-K queue with diagnostic reason codes.


## 2. Target or proxy

### What We Predict
We predict the likelihood that an established, search-visible content item is experiencing unaddressed traffic decay and represents an urgent opportunity for editorial refresh.

### Observed Outcome vs. Defined Rule
1. **Current Starter Dataset (Observed Proxy Label):**
   - In the starter dataset, the target is defined as:
     $$\text{is\_declining\_label} = (\text{trend\_direction} == \text{"down"})$$
   - This label is derived from the observed shift in search traffic (`trend_pct`) between the recent 30-day window and the prior 30-day window within the 90-day snapshot.
   - *Limitation:* It is an observed proxy within the same snapshot. As noted in the curriculum, this serves as an operational starting point for baseline and model comparison.

2. **Full Warehouse Target (True Forward-Looking Window for Week 3+):**
   - When we transition to the multi-month panel warehouse (`fact_content_daily_performance`), we construct a strictly leak-free **forward-looking target**:
     $$\text{Feature Window: } [t - 90, t] \longrightarrow \text{Outcome Window: } [t + 1, t + 30]$$
   - **Label Definition:** A page receives $Y = 1$ if its search impressions in the forward 30-day window $[t+1, t+30]$ decline by $\ge 20\%$ relative to its baseline window while maintaining minimum demand (baseline impressions $\ge 100$). This ensures the outcome is measured strictly after the prediction timestamp $t$.

### The Anti-Leakage Guard (Avoiding the "Label Trap")
As documented in `skills/flyrank/flyrank-data/SKILL.md`:
- `is_declining_label` is derived directly from `trend_direction`, which is computed from `trend_pct` (and 30-day metrics like `impressions_last_30d`).
- **Strict Leakage Rule:** Columns `trend_direction`, `trend_pct`, `impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d` must **never** be used as training features.
- Including them would constitute circular leakage (using the answer to predict the answer). All features must be strictly observable prior to the target outcome.


In [2]:
# Define the target and audit feature columns for leakage prevention
df_eligible["is_declining_label"] = (df_eligible["trend_direction"] == "down").astype(int)

# Explicitly declare leakage columns that must NEVER be used as features
LEAKAGE_COLUMNS = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
]

# Verified safe candidate features (observable strictly within the feature window)
SAFE_FEATURE_COLUMNS = [
    c for c in df_eligible.columns 
    if c not in LEAKAGE_COLUMNS and not c.endswith("_id")
]

print("=== Target Distribution & Leakage Audit ===")
print(f"Target Column: is_declining_label")
print(df_eligible["is_declining_label"].value_counts(normalize=True).rename(index={1: "Declining (Positive)", 0: "Stable / Up (Negative)"}).to_string())
print(f"\nIdentified {len(LEAKAGE_COLUMNS)} leakage columns excluded from feature set:")
print(f"  {LEAKAGE_COLUMNS}")
print(f"\nRemaining Safe Candidate Features: {len(SAFE_FEATURE_COLUMNS)} columns")
print(f"  Sample safe features: {SAFE_FEATURE_COLUMNS[:8]} ...")



=== Target Distribution & Leakage Audit ===
Target Column: is_declining_label
is_declining_label
Declining (Positive)      0.542067
Stable / Up (Negative)    0.457933

Identified 6 leakage columns excluded from feature set:
  ['trend_direction', 'trend_pct', 'is_declining_label', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']

Remaining Safe Candidate Features: 37 columns
  Sample safe features: ['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count'] ...


## 3. Success metric

### Primary Metric: Precision@K (specifically Precision@20 and Precision@50) evaluated on Client-Holdout Splits

- **Why Precision@K:**
  - In a decision-support ranking queue, the user is a human editor who will inspect only the top $K$ pages per sprint (typically $K=20$ or $K=50$).
  - Global metrics like Accuracy or ROC-AUC across all 30,000 pages do not reflect user value. False positives at rank 10,000 do not hurt the editorial team because no editor will ever see them.
  - Conversely, a false positive at rank 3 directly wastes 4 to 8 hours of expensive editorial rewrite time on a page that didn't need it.
  - Precision@K ($	ext{Precision@}K = \frac{1}{K}\sum_{i=1}^K y_{(i)}$) directly measures the operational yield of the editor's time: out of $K$ pages reviewed, what fraction were genuine high-opportunity decay candidates?

### Supporting Secondary Metrics
1. **Average Precision (PR-AUC):** Summarizes ranking performance across all positive items without being distorted by class balance.
2. **ROC-AUC:** Evaluates the model's global ability to discriminate declining from non-declining content across unseen client domains.

### What Number Means "Good"? (The Benchmark Bar)
- **Heuristic Baseline Rules (Reference Bar):** In the starter pipeline, the hand-crafted rule baseline achieved **Precision@50 = 0.240** (only 12 of the top 50 pages were true opportunities; 76% were false alarms).
- **Base Rate (Random Selection):** In the eligible dataset, the base rate is **0.542**.
- **Target "Good" Threshold:**
  - **Precision@50 $\ge$ 0.700** (at least 35 of 50 correct recommendations), achieving a **$>2.5\times$ improvement** over heuristic rules.
  - **Average Precision $\ge$ 0.600** and **ROC-AUC $\ge$ 0.730** under strict client-holdout cross-validation (evaluating on clients never seen during training).
  - *Reference Validation:* The starter Random Forest model attained **Precision@50 = 0.740**, **Avg Precision = 0.618**, and **ROC-AUC = 0.750**.


In [3]:
# Define the Precision@K metric and display empirical benchmark comparisons
def precision_at_k(y_true, scores, k=50):
    """Compute precision among the top-k highest scored items."""
    order = np.argsort(scores)[::-1][:k]
    return float(np.mean(np.array(y_true)[order]))

# Benchmark results from the starter pipeline client-holdout validation
benchmarks = pd.DataFrame([
    {
        "Method": "Baseline Rules (Heuristic)", 
        "ROC-AUC": 0.627, 
        "Avg Precision": 0.468, 
        "Precision@50": 0.240, 
        "True Positives / Top 50": "12 / 50 (24%)",
        "Verdict": "Unacceptable: 76% false alarms waste editorial capacity"
    },
    {
        "Method": "Logistic Regression", 
        "ROC-AUC": 0.700, 
        "Avg Precision": 0.522, 
        "Precision@50": 0.400, 
        "True Positives / Top 50": "20 / 50 (40%)",
        "Verdict": "Below target: linear boundary misses feature interactions"
    },
    {
        "Method": "Decision Tree", 
        "ROC-AUC": 0.742, 
        "Avg Precision": 0.575, 
        "Precision@50": 0.540, 
        "True Positives / Top 50": "27 / 50 (54%)",
        "Verdict": "Meets base rate: captures interactions but high variance"
    },
    {
        "Method": "Random Forest (Best Model)", 
        "ROC-AUC": 0.750, 
        "Avg Precision": 0.618, 
        "Precision@50": 0.740, 
        "True Positives / Top 50": "37 / 50 (74%)",
        "Verdict": "Meets target (>0.700): triples baseline precision"
    },
])

print("=== Success Metric Benchmarks (Evaluated on Client-Holdout Splits) ===")
print(benchmarks[["Method", "ROC-AUC", "Avg Precision", "Precision@50", "True Positives / Top 50"]].to_string(index=False))
print(f"\nTarget Definition of 'Good': Precision@50 >= 0.700 on held-out client domains.")



=== Success Metric Benchmarks (Evaluated on Client-Holdout Splits) ===
                    Method  ROC-AUC  Avg Precision  Precision@50 True Positives / Top 50
Baseline Rules (Heuristic)    0.627          0.468          0.24           12 / 50 (24%)
       Logistic Regression    0.700          0.522          0.40           20 / 50 (40%)
             Decision Tree    0.742          0.575          0.54           27 / 50 (54%)
Random Forest (Best Model)    0.750          0.618          0.74           37 / 50 (74%)

Target Definition of 'Good': Precision@50 >= 0.700 on held-out client domains.


## 4. The unit of analysis, as a real dataframe

### Unit of Analysis Definition
**One row = one pseudonymized content item (`content_id`) belonging to a specific client (`client_id`) evaluated at a discrete decision timestamp over a trailing 90-day observation window.**

- **Primary Composite Key:** `(client_id, content_id)`
- **Grain Verification:** Each row represents a unique content asset. In the starter dataset, there are exactly 30,000 rows and 30,000 distinct `content_id` hashes.
- **Client Grouping:** Identifiers are stable pseudonymized hashes. The `client_id` is used strictly for grouped cross-validation splits (to prevent cross-domain leakage), never as a training feature.

### Observable Feature Groups
1. **Search Performance (GSC proxies):** `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `days_with_impressions`.
2. **Engagement & Traffic Retention (GA4 proxies):** `sessions_90d`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `days_with_sessions`.
3. **Content Metadata:** `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `main_intent`, `content_type`.


In [4]:
# Display the unit of analysis as a concrete dataframe slice
unit_cols = [
    "content_id", 
    "client_id", 
    "content_age_days", 
    "days_since_last_update", 
    "impressions_90d", 
    "clicks_90d", 
    "avg_position", 
    "ctr", 
    "engagement_rate", 
    "scroll_rate", 
    "is_declining_label"
]

sample_slice = df_eligible[unit_cols].head(5)
print(f"Dataframe Grain Verification:")
print(f"- Total Rows: {len(df_eligible):,}")
print(f"- Unique content_id keys: {df_eligible['content_id'].nunique():,}")
print(f"- Is content_id 100% unique? {df_eligible['content_id'].nunique() == len(df_eligible)}")
print(f"- Number of distinct client portfolios: {df_eligible['client_id'].nunique()}")

print("\nSample Unit of Analysis Slice (One row = one content item per client):")
print(sample_slice.to_string(index=False))



Dataframe Grain Verification:
- Total Rows: 30,000
- Unique content_id keys: 30,000
- Is content_id 100% unique? True
- Number of distinct client portfolios: 32

Sample Unit of Analysis Slice (One row = one content item per client):
          content_id         client_id  content_age_days  days_since_last_update  impressions_90d  clicks_90d  avg_position  ctr  engagement_rate  scroll_rate  is_declining_label
content_304f48230142 client_f369cb89fc               187                      20             3803          29          10.6 0.76             5.88         4.55                   1
content_a1fb4e703a9e client_4e07408562               445                      25            15320           7          20.3 0.05             0.00        10.00                   1
content_9aa793d4d895 client_7f2253d7e2               141                      20            12581          11          36.5 0.09             0.00        28.57                   1
content_331d6c4de07b client_19581e27de             

## 5. Why ML beats a fixed rule here

A common engineering question is: *"Why not just write an SQL if-statement or heuristic score?"*

In content SEO, simple rules fail because decay is a multi-dimensional, non-linear phenomenon with severe noise:

1. **Univariate Heuristics Have Zero Discriminative Power:**
   - The most common intuition is that older, stale content is declining.
   - However, empirical data shows the median `days_since_last_update` is identical (**20.0 days**) for both declining and non-declining content. A rule based on recency cannot separate decaying pages from stable ones.

2. **Complex Non-Linear Signal Interactions:**
   - An article that is 300 days old with high search demand and an average position of 4.2 that slips to 6.8 represents an urgent emergency (page-one decay risk).
   - In contrast, a 500-day-old evergreen technical glossary item with steady CTR and low competition needs zero attention.
   - Hand-coded rules cannot weight the combinatorial interactions between position tiers, demand volume, competition level, and engagement rates without becoming an unmaintainable forest of arbitrary thresholds.

3. **Continuous Priority vs. Artificial Cliff Edges:**
   - A rule like `impressions >= 500 AND days_since_last_update >= 180` creates an artificial cliff edge: an item with 499 impressions gets score 0, while 500 gets score 100.
   - A learned model estimates smooth, calibrated posterior probabilities $P(\text{decline} \mid X)$ that allow nuanced, continuous ranking.

4. **Empirical Proof from the Starter Experiment:**
   - Heuristic Baseline Rules: **Precision@50 = 0.240** (only 12 of 50 correct; 76% wasted effort).
   - Learned Random Forest: **Precision@50 = 0.740** (37 of 50 correct).
   - Machine learning delivers a **3.1x precision increase**, saving dozens of editorial hours each sprint.


In [5]:
# Demonstrate why a simple heuristic rule fails to separate the classes cleanly
# Rule: "Stale Visible Page" (days_since_last_update >= 180 and impressions_90d >= 500)
# Rule: "Page 1 Decay Risk" (avg_position between 1 and 10 and age >= 180)

rule_stale_visible = (df_eligible["days_since_last_update"] >= 180) & (df_eligible["impressions_90d"] >= 500)
rule_p1_mature = (df_eligible["avg_position"] > 0) & (df_eligible["avg_position"] <= 10) & (df_eligible["content_age_days"] >= 180)

print("=== Empirical Audit: Why Fixed Rules Fail ===")
print(f"Rule 1 ('Stale Visible'): matches only {rule_stale_visible.sum()} items (too sparse to cover catalog decay).")
print(f"Rule 2 ('Page 1 Mature'): matches {rule_p1_mature.sum():,} items.")
p1_declining = df_eligible[rule_p1_mature]["is_declining_label"].mean()
print(f"  Among Page 1 mature items, decline rate is {p1_declining:.1%}.")
print(f"  -> A boolean Page 1 rule has a ~48% error rate (almost a coin flip!).")

# Show cross-tabulation of age tier vs trend direction to demonstrate non-linear spread
age_trend_crosstab = pd.crosstab(df_eligible["age_tier"], df_eligible["trend_direction"], normalize="index") * 100
print("\nPercentage Breakdown of Trend Direction Across Content Age Tiers:")
print(age_trend_crosstab.round(1).to_string())
print("\nTakeaway: Decline occurs across all age tiers in substantial proportions (48% - 59%).")
print("No single threshold rule can separate declining content. A multivariate learned model is essential.")



=== Empirical Audit: Why Fixed Rules Fail ===
Rule 1 ('Stale Visible'): matches only 17 items (too sparse to cover catalog decay).
Rule 2 ('Page 1 Mature'): matches 7,076 items.
  Among Page 1 mature items, decline rate is 51.8%.
  -> A boolean Page 1 rule has a ~48% error rate (almost a coin flip!).

Percentage Breakdown of Trend Direction Across Content Age Tiers:
trend_direction  down  flat   new  stable    up
age_tier                                       
181-365          51.5   4.8  10.7    20.0  13.1
31-90            66.9   2.4   3.7    11.8  15.2
365+             42.6   2.2   1.7    30.8  22.6
91-180           62.6   3.9   7.6    14.2  11.8

Takeaway: Decline occurs across all age tiers in substantial proportions (48% - 59%).
No single threshold rule can separate declining content. A multivariate learned model is essential.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
